In [22]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("../data/raw/demand.csv")

# Parse date
df["Date"] = pd.to_datetime(df["Date"])

# Clean Order_Demand
df["Order_Demand"] = (
    df["Order_Demand"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

df["Order_Demand"] = pd.to_numeric(df["Order_Demand"], errors="coerce")

df = (
    df.groupby(
        ["Product_Code", "Warehouse", "Product_Category", "Date"],
        as_index=False
    )["Order_Demand"]
    .sum()
)

df.duplicated(
    subset=["Product_Code", "Warehouse", "Date"]
).sum()

# Sort properly (CRITICAL)
df = df.sort_values(["Product_Code", "Warehouse", "Date"])

In [23]:
## Data Ordering for Time-Series Features

# The dataset is sorted by `Product_Code` and `Date` before creating lag features.

# This ensures that temporal operations like `shift()` correctly reference past demand values for the same product.
# Without proper sorting, lag features would be incorrect.

In [24]:
## Creating Lag Features

# Lag features allow the model to learn temporal dependencies.

# For each product, we create:

# - `lag_1`: Demand from previous day
# - `lag_7`: Demand from previous week
# - `lag_14`: Demand from two weeks ago

# These features help the model understand short-term and weekly demand behavior.

In [25]:
#creating lag features

df["lag_1"] = df.groupby(
    ["Product_Code", "Warehouse"]
)["Order_Demand"].shift(1)

df["lag_7"] = df.groupby(
    ["Product_Code", "Warehouse"]
)["Order_Demand"].shift(7)

df["lag_14"] = df.groupby(
    ["Product_Code", "Warehouse"]
)["Order_Demand"].shift(14)

In [26]:
df = df.dropna(subset = ["lag_1","lag_7","lag_14"])

In [27]:
## Why Grouping by Product_Code is Required

# Lag features are computed separately for each `Product_Code`.

# If grouping is not applied, demand values from different products would mix during the shift operation, causing data leakage.

# Grouping preserves product-level temporal behavior and ensures valid historical relationships.

In [28]:
## Rolling Window Features

# Rolling statistics summarize recent demand behavior.

# - `rolling_mean_7` captures short-term demand trend.
# - `rolling_std_7` captures short-term volatility.

# We shift before applying the rolling window to prevent using current-day information in feature computation, which would cause data leakage.

In [29]:
df["rolling_mean_7"] = (
    df.groupby(["Product_Code", "Warehouse"])["Order_Demand"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

df["rolling_std_7"] = (
    df.groupby(["Product_Code", "Warehouse"])["Order_Demand"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

In [30]:
df = df.dropna(subset=[
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_mean_7",
    "rolling_std_7"
])

In [31]:
df[["Product_Code", "Warehouse", "Date", "Order_Demand", "lag_1"]].head(10) 

,Product_Code,Warehouse,Date,Order_Demand,lag_1
21,Product_0001,Whse_A,2012-07-02,5000.0,2000.0
22,Product_0001,Whse_A,2012-07-04,1000.0,5000.0
23,Product_0001,Whse_A,2012-07-23,2000.0,1000.0
24,Product_0001,Whse_A,2012-08-03,200.0,2000.0
25,Product_0001,Whse_A,2012-08-07,1000.0,200.0
26,Product_0001,Whse_A,2012-08-09,3000.0,1000.0
27,Product_0001,Whse_A,2012-08-13,200.0,3000.0
28,Product_0001,Whse_A,2012-08-15,400.0,200.0
29,Product_0001,Whse_A,2012-08-21,600.0,400.0
30,Product_0001,Whse_A,2012-08-23,1000.0,600.0


In [32]:
#calender features

df["day_of_week"] = df["Date"].dt.dayofweek
df["month"] = df["Date"].dt.month
df["quarter"] = df["Date"].dt.quarter
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [34]:
## Calendar Features

# Calendar features help the model capture seasonal patterns such as weekly and monthly demand variations.